# Alexa PLN — Asistente de voz modular

Proyecto de examen (PLN). Asistente tipo Alexa con **módulos intercambiables**:
cada parte (STT, TTS, sentimiento, reconocer integrante, resumen) es una función
registrada en un diccionario. **Para cambiar de estrategia solo editas el diccionario
`CONFIG`** de la celda 2 — no tocas nada más.

| Componente | Opciones (`CONFIG`) |
|---|---|
| `stt` | `teclado` (sin audio, para probar) · `whisper` (faster-whisper local) |
| `tts` | `print` (sin audio) · `kokoro` · `piper` |
| `sentimiento` | `lexico` · `centroides` (TF-IDF + coseno) · `bert` |
| `reconocer` | `ngramas` (n-gramas de caracteres + coseno) · `exacto` |
| `resumen` | `textrank` (extractivo por oraciones) |

**Cómo trabajar:** ejecuta todas las celdas de definición de arriba hacia abajo,
luego usa las celdas de prueba para validar cada módulo por separado, y al final
corre la celda `correr_alexa()`. Empieza con todo en modo `teclado`/`print` y, cuando
funcione, cambia a `whisper`/`kokoro`.

Las partes de PLN del curso (tokenizar, n-gramas, TF-IDF, coseno, TextRank) están
implementadas a mano. La parte de orquestación (audio, estados) usa librerías normales.


In [69]:
# === INSTALACION (descomenta lo que necesites; corre una sola vez) ===
# Modo texto NO necesita nada salvo numpy.
# %pip install numpy

# Para voz (cuando ya funcione en modo teclado):
# %pip install faster-whisper sounddevice soundfile
# %pip install kokoro            # TTS calidad (revisa nombre de voz/idioma)
# %pip install piper-tts         # TTS respaldo, voces es_MX
# %pip install transformers torch   # solo si usas sentimiento "bert"


In [70]:
import numpy as np
import re
from datetime import datetime
from pathlib import Path


In [ ]:
# CONFIG  ->  cambia de estrategia escribiendo otro string
CONFIG = {
    "stt":         "whisper",    # "teclado" | "whisper"
    "tts":         "piper",      # "print"   | "kokoro" | "piper"
    "sentimiento": "lexico",     # "lexico"  | "centroides" | "bert"
    "reconocer":   "ngramas",    # "ngramas" | "exacto"
    "resumen":     "textrank",   # "textrank"
}


In [72]:
# LOG en tiempo real
def log(msg, tipo="info"):
    etq = {"info":"        ", "tu":"  [TU]   ", "alexa":"  [ALEXA]",
           "det":"  [DETEC]", "estado":">>ESTADO"}
    print(etq.get(tipo, "        ") + "  " + str(msg))


In [73]:
# PREPROCESAMIENTO (estilo del curso: sin split/lower/append)
def A_minusculas(texto):
    salida = ""
    for c in texto:
        o = ord(c)
        if o >= 65 and o <= 90:
            c = chr(o + 32)
        salida += c
    return salida

LETRAS = "abcdefghijklmnopqrstuvwxyzñáéíóúü"
def tokenizar(texto):
    texto = A_minusculas(texto)
    tokens = []
    actual = ""
    for c in texto:
        if c in LETRAS:
            actual += c
        else:
            if actual != "":
                tokens += [actual]
                actual = ""
    if actual != "":
        tokens += [actual]
    return tokens

STOPWORDS = ["de","la","que","el","en","y","a","los","del","se","las","por","un",
             "para","con","una","su","al","lo","es","mi","tu","ya","o","e","si",
             "muy","mas","más","son","como","han","ha","esta","este","estas"]
def quitar_stopwords(tokens):
    salida = []
    for t in tokens:
        if t not in STOPWORDS:
            salida += [t]
    return salida


In [74]:
# PRIMITIVAS DEL CURSO: coseno, n-gramas de caracteres, TF-IDF
def coseno(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def perfil_ngramas(texto, n=2):
    texto = A_minusculas(texto)
    perfil = {}
    i = 0
    while i <= len(texto) - n:
        g = texto[i:i+n]
        if g in perfil:
            perfil[g] += 1
        else:
            perfil[g] = 1
        i += 1
    return perfil

def coseno_perfiles(p1, p2):
    claves = {}
    for k in p1: claves[k] = True
    for k in p2: claves[k] = True
    dot = 0.0; s1 = 0.0; s2 = 0.0
    for k in claves:
        v1 = p1[k] if k in p1 else 0
        v2 = p2[k] if k in p2 else 0
        dot += v1 * v2; s1 += v1 * v1; s2 += v2 * v2
    if s1 == 0 or s2 == 0:
        return 0.0
    return dot / ((s1 ** 0.5) * (s2 ** 0.5))

def _vocabulario(corpus_tokens):
    vocab = []
    for toks in corpus_tokens:
        for t in toks:
            if t not in vocab:
                vocab += [t]
    return vocab

def tfidf_fit(corpus_tokens):
    vocab = _vocabulario(corpus_tokens)
    N = len(corpus_tokens)
    idf = {}
    for t in vocab:
        df = 0
        for toks in corpus_tokens:
            if t in toks:
                df += 1
        idf[t] = np.log((N + 1) / (df + 1)) + 1.0
    return vocab, idf

def tfidf_vector(tokens, vocab, idf):
    v = np.zeros(len(vocab))
    total = len(tokens)
    if total == 0:
        return v
    j = 0
    while j < len(vocab):
        t = vocab[j]
        c = 0
        for w in tokens:
            if w == t:
                c += 1
        v[j] = (c / total) * idf[t]
        j += 1
    return v


In [75]:
# MODULO: RECONOCER INTEGRANTE  (intercambiable)
INTEGRANTES = {
    "Edu":  "hola alexa",
    "Ana":  "alexa que onda",
    "Beto": "alexa que cuentas",
    "Caro": "alexa como estas",
    "Dani": "alexa que hay",
}
UMBRAL_SALUDO = 0.55

def rec_ngramas(texto_dicho):
    p = perfil_ngramas(texto_dicho, 2)
    mejor = None; mejor_s = -1.0
    for nombre in INTEGRANTES:
        s = coseno_perfiles(p, perfil_ngramas(INTEGRANTES[nombre], 2))
        if s > mejor_s:
            mejor_s = s; mejor = nombre
    if mejor_s >= UMBRAL_SALUDO:
        return mejor, mejor_s
    return None, mejor_s

def rec_exacto(texto_dicho):
    t = A_minusculas(texto_dicho)
    for nombre in INTEGRANTES:
        if INTEGRANTES[nombre] in t:
            return nombre, 1.0
    return None, 0.0


In [76]:
# MODULO: SENTIMIENTO  (intercambiable)
LEXICON = {
    "feliz":2,"contento":2,"contenta":2,"alegre":2,"genial":2,"increible":2,
    "bien":1,"bueno":1,"buena":1,"excelente":2,"tranquilo":1,"emocionado":2,
    "agradecido":1,"motivado":1,"perfecto":2,"amor":2,"divertido":2,
    "relajado":1,"orgulloso":2,"maravilloso":2,"descansado":1,
    "triste":-2,"mal":-1,"malo":-1,"mala":-1,"cansado":-1,"cansada":-1,
    "estresado":-2,"estresada":-2,"enojado":-2,"molesto":-1,"horrible":-2,
    "terrible":-2,"deprimido":-2,"preocupado":-1,"aburrido":-1,"frustrado":-2,
    "agotado":-2,"miedo":-1,"solo":-1,"pesimo":-2,
}
NEGACIONES = ["no","nunca","tampoco","ni"]

def sent_lexico(texto):
    tokens = tokenizar(texto)
    score = 0
    i = 0
    while i < len(tokens):
        t = tokens[i]
        if t in LEXICON:
            val = LEXICON[t]
            if i > 0 and tokens[i-1] in NEGACIONES:
                val = -val
            score += val
        i += 1
    if score > 0: return "positivo"
    if score < 0: return "negativo"
    return "neutral"

EJEMPLOS_POS = [
    "estoy muy feliz y contento hoy", "fue un dia genial y divertido",
    "me siento increible y motivado", "todo salio perfecto y estoy agradecido",
    "que dia tan maravilloso y tranquilo", "estoy emocionado y de buen humor",
    "me fue excelente y estoy orgulloso", "paso algo bueno y me alegra mucho",
]
EJEMPLOS_NEG = [
    "estoy muy triste y cansado", "fue un dia horrible y estresante",
    "me siento agotado y deprimido", "todo salio mal y estoy frustrado",
    "que dia tan terrible y aburrido", "estoy preocupado y de mal humor",
    "me fue pesimo y tengo miedo", "paso algo malo y me siento solo",
]
_C = {}
def _init_centroides():
    pos = [tokenizar(f) for f in EJEMPLOS_POS]
    neg = [tokenizar(f) for f in EJEMPLOS_NEG]
    todos = pos + neg
    vocab, idf = tfidf_fit(todos)
    Vp = [tfidf_vector(t, vocab, idf) for t in pos]
    Vn = [tfidf_vector(t, vocab, idf) for t in neg]
    _C["vocab"] = vocab; _C["idf"] = idf
    _C["pos"] = np.mean(Vp, axis=0); _C["neg"] = np.mean(Vn, axis=0)

def sent_centroides(texto):
    if "vocab" not in _C:
        _init_centroides()
    v = tfidf_vector(tokenizar(texto), _C["vocab"], _C["idf"])
    sp = coseno(v, _C["pos"]); sn = coseno(v, _C["neg"])
    if sp == 0 and sn == 0:
        return "neutral"
    if abs(sp - sn) < 0.02:
        return "neutral"
    return "positivo" if sp > sn else "negativo"

_BERT = {}
def sent_bert(texto):
    try:
        if "pipe" not in _BERT:
            from transformers import pipeline
            _BERT["pipe"] = pipeline(
                "sentiment-analysis",
                model="pysentimiento/robertuito-sentiment-analysis")
        r = _BERT["pipe"](texto)[0]
        etiqueta = r["label"].upper()
        if etiqueta in ("POS", "POSITIVE"): return "positivo"
        if etiqueta in ("NEG", "NEGATIVE"): return "negativo"
        return "neutral"
    except Exception as e:
        log("BERT no disponible, uso lexico. (" + str(e)[:40] + ")")
        return sent_lexico(texto)


In [77]:
# MODULO: DATOS CURIOSOS  (busqueda hibrida)
DATOS_ANIO = {
    1969:"en 1969 el ser humano piso la Luna por primera vez.",
    1989:"en 1989 cayo el Muro de Berlin.",
    1991:"en 1991 se publico la primera pagina web de la historia.",
    1994:"en 1994 nacio el formato de imagen que hoy llamamos JPEG estandarizado.",
    2000:"en el año 2000 el mundo temio el famoso error Y2K.",
    2004:"en 2004 se fundo una red social que cambiaria internet.",
    2007:"en 2007 se presento el primer iPhone.",
    2012:"en 2012 una sonda llego a Marte llamada Curiosity.",
    2020:"en 2020 el mundo vivio una pandemia que acelero el trabajo remoto.",
}
def dato_anio(anio):
    if anio in DATOS_ANIO:
        return DATOS_ANIO[anio]
    mejor = None; dist = None
    for a in DATOS_ANIO:
        d = abs(a - anio)
        if dist is None or d < dist:
            dist = d; mejor = a
    if mejor is None:
        return "no tengo un dato para ese año."
    return DATOS_ANIO[mejor]

DATOS_PASATIEMPO = [
    {"clave":"futbol deporte balon cancha", "dato":"el futbol es el deporte mas visto del planeta."},
    {"clave":"musica tocar guitarra cantar", "dato":"escuchar musica libera dopamina, igual que la comida rica."},
    {"clave":"videojuegos jugar consola gaming", "dato":"la industria de los videojuegos factura mas que el cine y la musica juntos."},
    {"clave":"leer libros lectura novela", "dato":"leer seis minutos al dia reduce el estres hasta dos tercios."},
    {"clave":"cocinar cocina recetas comida", "dato":"cocinar en casa activa zonas del cerebro ligadas a la creatividad."},
    {"clave":"correr running ejercicio gym", "dato":"correr libera endorfinas, por eso existe la euforia del corredor."},
    {"clave":"pintar dibujar arte", "dato":"dibujar mejora la memoria mas que escribir notas."},
]
DATOS_GUSTO = [
    {"clave":"cafe te bebida", "dato":"el cafe fue descubierto, segun la leyenda, por unas cabras inquietas."},
    {"clave":"chocolate dulce postre", "dato":"el chocolate fue usado como moneda por los aztecas."},
    {"clave":"perros gatos mascotas animales", "dato":"los perros pueden aprender mas de 150 palabras."},
    {"clave":"cine peliculas series", "dato":"la primera pelicula de la historia duraba menos de un minuto."},
    {"clave":"viajar viajes playa montaña", "dato":"viajar a lugares nuevos genera nuevas conexiones neuronales."},
    {"clave":"tecnologia computadoras gadgets", "dato":"la primera computadora pesaba mas de 27 toneladas."},
    {"clave":"naturaleza plantas jardin", "dato":"las plantas se comunican entre si por compuestos quimicos."},
]
def dato_semantico(texto, dataset):
    p = perfil_ngramas(texto, 3)
    mejor = None; mejor_s = -1.0
    for item in dataset:
        s = coseno_perfiles(p, perfil_ngramas(item["clave"], 3))
        if s > mejor_s:
            mejor_s = s; mejor = item
    if mejor is None:
        return "no tengo un dato relacionado."
    return mejor["dato"]

def extraer_anio(texto):
    m = re.search(r"\b(?:19|20)\d{2}\b", texto)
    if m:
        return int(m.group(0))
    return None


In [78]:
# MODULO: RESUMEN  (extractivo por oraciones = fluido)
def partir_oraciones(texto):
    oraciones = []
    actual = ""
    for c in texto:
        actual += c
        if c == "." or c == "!" or c == "?":
            o = actual.strip()
            if len(o) > 0:
                oraciones += [o]
            actual = ""
    o = actual.strip()
    if len(o) > 0:
        oraciones += [o]
    return oraciones

def resumir_textrank(texto, n=3):
    oraciones = partir_oraciones(texto)
    if len(oraciones) <= n:
        return texto.strip()
    toks = [quitar_stopwords(tokenizar(o)) for o in oraciones]
    vocab, idf = tfidf_fit(toks)
    V = [tfidf_vector(t, vocab, idf) for t in toks]
    m = len(oraciones)
    S = np.zeros((m, m))
    for i in range(m):
        for j in range(m):
            if i != j:
                S[i][j] = coseno(V[i], V[j])
    scores = np.ones(m) / m
    d = 0.85
    for _ in range(40):
        nuevos = np.ones(m) * (1 - d) / m
        for i in range(m):
            for j in range(m):
                grado = S[j].sum()
                if grado > 0:
                    nuevos[i] += d * (S[j][i] / grado) * scores[j]
        scores = nuevos
    orden = list(np.argsort(scores)[::-1][:n])
    orden.sort()
    resumen = ""
    for k in orden:
        resumen += oraciones[k] + " "
    return resumen.strip()


In [79]:
# SALUDO segun hora + clima
def franja_horaria():
    h = datetime.now().hour
    if h < 12: return "Buenos dias"
    if h < 19: return "Buenas tardes"
    return "Buenas noches"

def clima_actual(lat=19.43, lon=-99.13):
    try:
        import urllib.request, json
        url = ("https://api.open-meteo.com/v1/forecast?latitude="
               + str(lat) + "&longitude=" + str(lon) + "&current=temperature_2m")
        with urllib.request.urlopen(url, timeout=4) as r:
            d = json.loads(r.read())
        return "hay " + str(d["current"]["temperature_2m"]) + " grados"
    except Exception:
        return None

def construir_saludo(nombre):
    s = franja_horaria()
    c = clima_actual()
    if c:
        return s + ", " + nombre + ". Ahora mismo " + c + " por aqui. Cuentame, como estuvo tu dia?"
    return s + ", " + nombre + ". Cuentame, como estuvo tu dia?"


In [ ]:
# MODULO: STT y TTS  (intercambiables)
def stt_teclado(prompt="> "):
    return input(prompt)

_WHISPER = {}
def stt_whisper(prompt=None, segundos=6, fs=16000):
    import sounddevice as sd
    from faster_whisper import WhisperModel
    if "m" not in _WHISPER:
        _WHISPER["m"] = WhisperModel("small", device="cpu", compute_type="int8")
    log("...escuchando " + str(segundos) + "s...")
    audio = sd.rec(int(segundos * fs), samplerate=fs, channels=1, dtype="float32")
    sd.wait()
    audio = audio.reshape(-1)
    segs, _ = _WHISPER["m"].transcribe(audio, language="es", vad_filter=True)
    texto = ""
    for s in segs:
        texto += s.text
    return texto.strip()

def tts_print(texto):
    log(texto, "alexa")

_KOKORO = {}
def tts_kokoro(texto):
    log(texto, "alexa")
    import sounddevice as sd
    from kokoro import KPipeline
    if "p" not in _KOKORO:
        _KOKORO["p"] = KPipeline(lang_code="e")   # 'e' = español
    for _, _, audio in _KOKORO["p"](texto, voice="ef_dora", speed=1.0):
        sd.play(audio, 24000); sd.wait()

def tts_piper(texto):
    log(texto, "alexa")
    import subprocess, soundfile as sf, sounddevice as sd
    wav = "/tmp/_alexa.wav"
    subprocess.run(["piper", "--model", "es_MX-claude-high.onnx",
                    "--output_file", wav], input=texto.encode("utf-8"))
    data, sr = sf.read(wav)
    sd.play(data, sr); sd.wait()

VOZ_PIPER = "es_MX-claude-high"          # alternativa: "es_ES-davefx-medium"
DIR_VOCES = Path("voces_piper")          # carpeta local junto al notebook = tu cache
DIR_VOCES.mkdir(exist_ok=True)

_PIPER = {}
def cargar_piper(voz=VOZ_PIPER):
    from piper import PiperVoice
    from piper.download_voices import download_voice
    if "v" not in _PIPER:
        modelo = DIR_VOCES / (voz + ".onnx")
        if not modelo.exists():
            log("Descargando voz '" + voz + "' (solo la 1a vez, puede tardar)...", "det")
            download_voice(voz, DIR_VOCES)
        else:
            log("Voz '" + voz + "' ya en cache, cargando...", "det")
        _PIPER["v"] = PiperVoice.load(str(modelo))
    return _PIPER["v"]

def tts_piper(texto):
    log(texto, "alexa")
    import sounddevice as sd
    voz = cargar_piper()
    trozos = [chunk.audio_int16_array for chunk in voz.synthesize(texto)]
    audio = np.concatenate(trozos)
    sd.play(audio, voz.config.sample_rate)
    sd.wait()


In [ ]:
# DISPATCHERS  (leen CONFIG)
STT  = {"teclado": stt_teclado, "whisper": stt_whisper}
TTS  = {"print": tts_print, "kokoro": tts_kokoro, "piper": tts_piper}
SENT = {"lexico": sent_lexico, "centroides": sent_centroides, "bert": sent_bert}
REC  = {"ngramas": rec_ngramas, "exacto": rec_exacto}
RES  = {"textrank": resumir_textrank}

def escuchar(prompt="> "):
    return STT[CONFIG["stt"]](prompt)
def hablar(texto):
    TTS[CONFIG["tts"]](texto)
def analizar_sentimiento(texto):
    return SENT[CONFIG["sentimiento"]](texto)
def reconocer_integrante(texto):
    return REC[CONFIG["reconocer"]](texto)
def resumir(texto, n=4):
    return RES[CONFIG["resumen"]](texto, n)


In [82]:
# RESPUESTAS y AYUDANTES de conversacion
def respuesta_sentimiento(sent, usuario):
    if sent == "positivo":
        return "Me alegra mucho, " + usuario + "! Suena a un buen dia."
    if sent == "negativo":
        return "Lamento que no haya sido un buen dia, " + usuario + ". Espero que mejore pronto."
    return "Gracias por contarme, " + usuario + "."

def es_negacion(texto):
    t = A_minusculas(texto)
    for w in ["no gracias", "no, gracias", "ya no", "adios", "salir", "terminar"]:
        if w in t:
            return True
    tokens = tokenizar(texto)
    return ("no" in tokens) and ("si" not in tokens)

def es_afirmacion(texto):
    tokens = tokenizar(texto)
    for w in ["si", "claro", "va", "dale", "adelante", "bueno", "obvio"]:
        if w in tokens:
            return True
    return False


In [83]:
# CUENTO  (reemplaza por el tuyo de >= 1000 palabras)
CUENTO = """
El viejo faro se alzaba al final del acantilado desde hacia mas de cien años.
Marina llego al pueblo buscando silencio para terminar de escribir su novela.
La gente del lugar le advirtio que nadie vivia cerca del faro desde hacia decadas.
Decian que las noches de tormenta una luz se encendia sola en lo alto de la torre.
Marina no creia en fantasmas, asi que alquilo la casa mas cercana al acantilado.
La primera semana fue tranquila y avanzo mucho en su historia.
Una noche de viento fuerte, Marina vio la luz del faro encenderse en la oscuridad.
Curiosa y un poco asustada, decidio subir por el sendero hasta la torre.
Dentro encontro a un anciano que cuidaba la lampara con manos temblorosas.
El hombre le conto que habia sido el ultimo farero antes de que cerraran el faro.
Cada tormenta regresaba para encender la luz, porque temia que un barco se perdiera.
Marina comprendio que aquella costumbre era su forma de seguir siendo util.
Conmovida, le ofrecio ayuda para mantener el faro encendido durante el invierno.
El anciano sonrio por primera vez en muchos años y acepto la compañia.
Desde entonces, cada noche de tormenta, dos luces brillaban en el acantilado.
Una era la del faro y la otra la de la ventana donde Marina escribia.
El pueblo dejo de hablar de fantasmas y empezo a hablar de la escritora y el farero.
La novela de Marina se publico al año siguiente y se titulo precisamente El faro.
En la dedicatoria escribio que la soledad a veces solo necesita una luz al lado.
El anciano murio en paz aquel verano, pero la luz del faro nunca volvio a apagarse.
Marina se quedo a vivir en el pueblo y se encargo de cuidar la lampara cada tormenta.
"""


In [84]:
# MAQUINA DE ESTADOS
_ctx = {}
def correr_alexa():
    estado = "ESPERANDO_SALUDO"
    usuario = None
    while True:
        log(estado, "estado")
        if estado == "ESPERANDO_SALUDO":
            dicho = escuchar("(di tu saludo) > ")
            if dicho.strip() == "":
                continue
            nombre, score = reconocer_integrante(dicho)
            log("saludo='" + dicho + "'  match=" + str(nombre) + "  score=" + str(round(score, 2)), "det")
            if nombre is None:
                continue                      # no responde si no es un saludo valido
            usuario = nombre
            hablar(construir_saludo(usuario))
            estado = "DESCRIBIR_DIA"

        elif estado == "DESCRIBIR_DIA":
            desc = escuchar("(describe tu dia) > ")
            sent = analizar_sentimiento(desc)
            log("sentimiento=" + sent, "det")
            hablar(respuesta_sentimiento(sent, usuario))
            estado = "PEDIR_DATOS"

        elif estado == "PEDIR_DATOS":
            hablar("Dime un año, tu pasatiempo favorito y algo que te guste, todo junto.")
            datos = escuchar("(año, pasatiempo, gusto) > ")
            _ctx["anio"] = extraer_anio(datos)
            _ctx["datos"] = datos
            log("año=" + str(_ctx["anio"]), "det")
            estado = "DATO_CURIOSO"

        elif estado == "DATO_CURIOSO":
            anio = _ctx.get("anio")
            datos = _ctx.get("datos", "")
            if anio is not None:
                hablar("Sobre " + str(anio) + ": " + dato_anio(anio))
            hablar("Sobre tu pasatiempo: " + dato_semantico(datos, DATOS_PASATIEMPO))
            hablar("Y algo curioso: " + dato_semantico(datos, DATOS_GUSTO))
            estado = "OFRECER_RESUMEN"

        elif estado == "OFRECER_RESUMEN":
            hablar("Quieres escuchar el resumen del cuento? Si no, di 'No, gracias' para terminar.")
            r = escuchar("(si / no, gracias) > ")
            if es_negacion(r):
                hablar("De acuerdo, " + usuario + ". Hasta luego!")
                estado = "ESPERANDO_SALUDO"; usuario = None
            elif es_afirmacion(r):
                estado = "REPRODUCIR_RESUMEN"
            else:
                hablar("No te entendi, repite por favor.")

        elif estado == "REPRODUCIR_RESUMEN":
            resumen = resumir(CUENTO, 4)
            log("resumen de " + str(len(tokenizar(resumen))) + " palabras", "det")
            hablar(resumen)
            hablar("Eso es todo, " + usuario + ". Hasta luego!")
            estado = "ESPERANDO_SALUDO"; usuario = None


## Celdas de prueba por módulo

Ejecuta estas para validar cada pieza sin lanzar toda la Alexa.


In [85]:
# PRUEBA: reconocer integrante por el saludo
for s in ["alexa que onda", "hola alexa", "buenas computadora"]:
    print(s, "->", reconocer_integrante(s))


alexa que onda -> ('Ana', 1.0000000000000002)
hola alexa -> ('Edu', 1.0)
buenas computadora -> (None, 0.3429971702850177)


In [86]:
# PRUEBA: sentimiento (cambia CONFIG["sentimiento"] entre lexico/centroides)
for f in ["hoy fue un dia genial y feliz", "estoy triste y agotado", "comi y dormi"]:
    print(repr(f), "->", analizar_sentimiento(f))


'hoy fue un dia genial y feliz' -> positivo
'estoy triste y agotado' -> negativo
'comi y dormi' -> neutral


In [87]:
# PRUEBA: dato curioso (año + pasatiempo + gusto)
print(extraer_anio("nací en 2003 y me gusta el futbol"))
print(dato_anio(2007))
print(dato_semantico("me gustan los videojuegos", DATOS_PASATIEMPO))
print(dato_semantico("me encanta el chocolate", DATOS_GUSTO))


2003
en 2007 se presento el primer iPhone.
la industria de los videojuegos factura mas que el cine y la musica juntos.
el chocolate fue usado como moneda por los aztecas.


In [88]:
# PRUEBA: resumen extractivo (cambia n para mas/menos oraciones)
print(resumir(CUENTO, 4))


El viejo faro se alzaba al final del acantilado desde hacia mas de cien años. Una noche de viento fuerte, Marina vio la luz del faro encenderse en la oscuridad. Desde entonces, cada noche de tormenta, dos luces brillaban en el acantilado. Marina se quedo a vivir en el pueblo y se encargo de cuidar la lampara cada tormenta.


## Ejecutar la Alexa

La celda de abajo entra en un bucle infinito (es un asistente). **Para detenerlo:
botón Interrumpir / Kernel → Interrupt.** En modo `teclado` responde escribiendo;
en modo `whisper` habla cuando veas `...escuchando...`.

Saludos registrados (uno por integrante): `hola alexa`, `alexa que onda`,
`alexa que cuentas`, `alexa como estas`, `alexa que hay`. Cámbialos en la celda
de *RECONOCER INTEGRANTE*.


In [89]:
# Asegura modo de prueba (sin audio). Cambia cuando quieras voz real.
CONFIG["stt"] = "whisper"   # "whisper"
CONFIG["tts"] = "piper"     # "kokoro" / "piper"
correr_alexa()


>>ESTADO  ESPERANDO_SALUDO


c:\Users\josed\Documents\NLP\Alexa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


          ...escuchando 6s...
  [DETEC]  saludo='¡Hola Alexa!'  match=Edu  score=0.9
  [ALEXA]  Buenas tardes, Edu. Ahora mismo hay 20.8 grados por aqui. Cuentame, como estuvo tu dia?
  [DETEC]  Voz 'es_MX-claude-high' ya en cache, cargando...
>>ESTADO  DESCRIBIR_DIA
          ...escuchando 6s...
  [DETEC]  sentimiento=neutral
  [ALEXA]  Gracias por contarme, Edu.
>>ESTADO  PEDIR_DATOS
  [ALEXA]  Dime un año, tu pasatiempo favorito y algo que te guste, todo junto.
          ...escuchando 6s...
  [DETEC]  año=2000
>>ESTADO  DATO_CURIOSO
  [ALEXA]  Sobre 2000: en el año 2000 el mundo temio el famoso error Y2K.
  [ALEXA]  Sobre tu pasatiempo: leer seis minutos al dia reduce el estres hasta dos tercios.
  [ALEXA]  Y algo curioso: la primera pelicula de la historia duraba menos de un minuto.
>>ESTADO  OFRECER_RESUMEN
  [ALEXA]  Quieres escuchar el resumen del cuento? Si no, di 'No, gracias' para terminar.
          ...escuchando 6s...
  [ALEXA]  No te entendi, repite por favor.
>>ESTADO  OF

KeyboardInterrupt: 